## MLP - Actividades de modificación

En esta notebook se evalúan las modificaciones pedidas en `Preguntas.md`:

- Dropout;
- Batch Normalization;
- Weight Decay;
- Data Augmentation;
- Early Stopping;
- inicialización Xavier;
- inicialización He/Kaiming;
- inicialización uniforme;
- logging con TensorBoard;
- visualización de histogramas de pesos.

El test fijo se mantiene excluido. Todas las comparaciones se hacen con 5-fold cross-validation sobre `trainval`.

In [1]:
# ============================================================
# 1. Configuración de paths
# ============================================================

from pathlib import Path
import sys
import os
import json
import pandas as pd

REPO_ROOT = Path.cwd()

if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.append(str(REPO_ROOT / "src"))

print(f"Repo root: {REPO_ROOT}")

# ============================================================
# 2. Importar funciones de entrenamiento
# ============================================================

from mlp_training import run_cross_validation, save_experiment_outputs

Repo root: c:\Users\tomas\OneDrive\Documentos\MATERIAS ITBA\ELECTIVAS - CUATRIMESTRE X\MACHINE LEARNING y REDES NEURONALES EN BIOINGENIERÍA\skin-dataset-classification


In [2]:
# ============================================================
# 3. Definición de experimentos
# ============================================================
#
# La lógica es progresiva y está pensada para estudiar el efecto de
# distintas técnicas de regularización y ajuste de hiperparámetros.
#
# MLP_1:
#   Agrega Dropout respecto del baseline.
#
# MLP_2:
#   Agrega Batch Normalization además de Dropout.
#
# MLP_3:
#   Agrega Weight Decay además de Dropout + BatchNorm.
#   En la corrida anterior fue el mejor MLP (~62% accuracy media).
#
# MLP_4:
#   Agrega Data Augmentation sobre la configuración regularizada.
#
# MLP_5, MLP_6, MLP_7:
#   Comparan inicializaciones manuales:
#   Xavier, He/Kaiming y uniforme.
#
# MLP_8:
#   Prueba Early Stopping.
#
# MLP_9, MLP_10, MLP_11:
#   Configuraciones adicionales para intentar mejorar el desempeño:
#   - menor Dropout: 0.3 en lugar de 0.5;
#   - menor learning rate en algunos casos;
#   - mayor weight decay en un caso;
#   - red más grande en MLP_11.
#
# Todos los experimentos:
#   - usan el mismo split final;
#   - usan validación cruzada de 5 folds sobre trainval;
#   - NO usan el test fijo para elegir hiperparámetros;
#   - registran TensorBoard en runs/mlp/;
#   - registran MLflow en mlruns/.
# ============================================================

experiments = [
    {
        "experiment": "MLP_1_dropout",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": False,
        "weight_decay": 0.0,
        "augmentation": "minimal",

        "initialization": "default",
        "hidden_dims": [512, 128],

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_2_dropout_batchnorm",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 0.0,
        "augmentation": "minimal",

        "initialization": "default",
        "hidden_dims": [512, 128],

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_3_dropout_batchnorm_weightdecay",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "minimal",

        "initialization": "default",
        "hidden_dims": [512, 128],

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_4_data_augmentation",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "default",
        "hidden_dims": [512, 128],

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_5_xavier_initialization",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "xavier",
        "hidden_dims": [512, 128],

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_6_he_initialization",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "he",
        "hidden_dims": [512, 128],

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_7_uniform_initialization",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "uniform",
        "hidden_dims": [512, 128],

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_8_early_stopping",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 30,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "he",
        "hidden_dims": [512, 128],

        "early_stopping": True,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_9_dropout03_batchnorm_weightdecay",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 25,
        "lr": 1e-3,
        "optimizer": "adam",

        "dropout": 0.3,
        "batch_norm": True,
        "weight_decay": 1e-3,
        "augmentation": "minimal",

        "initialization": "he",
        "hidden_dims": [512, 128],

        "early_stopping": True,
        "patience": 5,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_10_lower_lr_dropout03_bn_wd",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 30,
        "lr": 3e-4,
        "optimizer": "adam",

        "dropout": 0.3,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "minimal",

        "initialization": "he",
        "hidden_dims": [512, 128],

        "early_stopping": True,
        "patience": 6,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_11_bigger_bn_dropout03",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 30,
        "lr": 3e-4,
        "optimizer": "adam",

        "dropout": 0.3,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "minimal",

        "initialization": "he",
        "hidden_dims": [1024, 256],

        "early_stopping": True,
        "patience": 6,

        "tensorboard": True,
        "mlflow": True,
        "log_pytorch_model": True,
        "histogram_every": 5,
        "seed": 42,
    },
]

In [3]:
# ============================================================
# 4. Ejecutar experimentos
# ============================================================
#
# Cada experimento corre 5 folds.
#
# Esto puede tardar más que el baseline, pero sigue siendo MLP,
# no CNN. Si una corrida se interrumpe, se puede volver a correr
# esta celda; los archivos se sobrescriben.
# ============================================================

all_outputs = []
summary_rows = []

for config in experiments:
    print("\n" + "#" * 90)
    print(f"Ejecutando experimento: {config['experiment']}")
    print("#" * 90)

    output = run_cross_validation(
        config=config,
        split_csv="data/splits/final_split_5fold.csv",
    )

    all_outputs.append(output)
    summary_rows.append(output["summary"])

    save_experiment_outputs(
        cv_output=output,
        output_prefix=config["experiment"].lower(),
    )


##########################################################################################
Ejecutando experimento: MLP_1_dropout
##########################################################################################
Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
[MLP_1_dropout] fold 0 | epoch 01/20 | train_acc=0.178 | val_acc=0.208 | val_f1=0.183
[MLP_1_dropout] fold 0 | epoch 02/20 | train_acc=0.248 | val_acc=0.368 | val_f1=0.309
[MLP_1_dropout] fold 0 | epoch 03/20 | train_acc=0.283 | val_acc=0.340 | val_f1=0.301
[MLP_1_dropout] fold 0 | epoch 04/20 | train_acc=0.279 | val_acc=0.361 | val_f1=0.311
[MLP_1_dropout] fold 0 | epoch 05/20 | train_acc=0.330 | val_acc=0.389 | val_f1=0.330
[MLP_1_dropout] fold 0 | epoch 06/20 | train_acc=0.332 | val_acc=0.403 | v

2026/06/17 15:49:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Archivos guardados:
- experiments\mlp_1_dropout_fold_results.csv
- experiments\mlp_1_dropout_history.csv
- experiments\mlp_1_dropout_summary.json
- results\training_curves\mlp_1_dropout_loss.png
- results\training_curves\mlp_1_dropout_accuracy.png
- results\confusion_matrices\mlp_1_dropout.png
- results\classification_reports\mlp_1_dropout_classification_report.txt

##########################################################################################
Ejecutando experimento: MLP_2_dropout_batchnorm
##########################################################################################
Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
[MLP_2_dropout_batchnorm] fold 0 | epoch 01/20 | train_acc=0.277 | val_acc=0.382 | val_f1=0.301
[MLP_2_dropout_batchnorm] fold

2026/06/17 16:07:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Archivos guardados:
- experiments\mlp_2_dropout_batchnorm_fold_results.csv
- experiments\mlp_2_dropout_batchnorm_history.csv
- experiments\mlp_2_dropout_batchnorm_summary.json
- results\training_curves\mlp_2_dropout_batchnorm_loss.png
- results\training_curves\mlp_2_dropout_batchnorm_accuracy.png
- results\confusion_matrices\mlp_2_dropout_batchnorm.png
- results\classification_reports\mlp_2_dropout_batchnorm_classification_report.txt

##########################################################################################
Ejecutando experimento: MLP_3_dropout_batchnorm_weightdecay
##########################################################################################
Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
[MLP_3_dropout_batchnorm_weightdecay] fold 0

...

Archivos guardados:
- experiments\mlp_9_dropout03_batchnorm_weightdecay_fold_results.csv
- experiments\mlp_9_dropout03_batchnorm_weightdecay_history.csv
- experiments\mlp_9_dropout03_batchnorm_weightdecay_summary.json
- results\training_curves\mlp_9_dropout03_batchnorm_weightdecay_loss.png
- results\training_curves\mlp_9_dropout03_batchnorm_weightdecay_accuracy.png
- results\confusion_matrices\mlp_9_dropout03_batchnorm_weightdecay.png
- results\classification_reports\mlp_9_dropout03_batchnorm_weightdecay_classification_report.txt

##########################################################################################
Ejecutando experimento: MLP_10_lower_lr_dropout03_bn_wd
##########################################################################################
Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'T

2026/06/17 18:38:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Archivos guardados:
- experiments\mlp_10_lower_lr_dropout03_bn_wd_fold_results.csv
- experiments\mlp_10_lower_lr_dropout03_bn_wd_history.csv
- experiments\mlp_10_lower_lr_dropout03_bn_wd_summary.json
- results\training_curves\mlp_10_lower_lr_dropout03_bn_wd_loss.png
- results\training_curves\mlp_10_lower_lr_dropout03_bn_wd_accuracy.png
- results\confusion_matrices\mlp_10_lower_lr_dropout03_bn_wd.png
- results\classification_reports\mlp_10_lower_lr_dropout03_bn_wd_classification_report.txt

##########################################################################################
Ejecutando experimento: MLP_11_bigger_bn_dropout03
##########################################################################################
Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion

2026/06/17 18:56:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Archivos guardados:
- experiments\mlp_11_bigger_bn_dropout03_fold_results.csv
- experiments\mlp_11_bigger_bn_dropout03_history.csv
- experiments\mlp_11_bigger_bn_dropout03_summary.json
- results\training_curves\mlp_11_bigger_bn_dropout03_loss.png
- results\training_curves\mlp_11_bigger_bn_dropout03_accuracy.png
- results\confusion_matrices\mlp_11_bigger_bn_dropout03.png
- results\classification_reports\mlp_11_bigger_bn_dropout03_classification_report.txt


In [4]:
# ============================================================
# 5. Cargar resumen del baseline
# ============================================================
#
# El baseline se entrenó en 01_MLP_baseline.ipynb.
# Para comparar todo junto, cargamos su summary.json.
# ============================================================

baseline_summary_path = Path("experiments/mlp_0_baseline_summary.json")

if baseline_summary_path.exists():
    with open(baseline_summary_path, "r", encoding="utf-8") as f:
        baseline_summary = json.load(f)

    all_summaries = [baseline_summary] + summary_rows

else:
    print("No se encontró el baseline. Se comparan solo las modificaciones.")
    all_summaries = summary_rows

In [5]:
# ============================================================
# 6. Tabla comparativa
# ============================================================
#
# Ordenamos por val_accuracy_mean.
# También conviene mirar macro_f1 y balanced_accuracy, porque puede
# haber desbalance de clases.
# ============================================================

summary_df = pd.DataFrame(all_summaries)

summary_df = summary_df.sort_values(
    by="val_accuracy_mean",
    ascending=False,
).reset_index(drop=True)

display(summary_df)

# ============================================================
# 7. Guardar comparación final de MLP
# ============================================================

output_csv = Path("experiments/experiments_mlp.csv")
summary_df.to_csv(output_csv, index=False)

print(f"Comparación MLP guardada en: {output_csv}")

,experiment,model,image_size,batch_size,epochs,lr,optimizer,dropout,batch_norm,weight_decay,...,early_stopping,folds_run,best_fold,val_accuracy_mean,val_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,mlflow_run_id
0,MLP_10_lower_lr_dropout03_bn_wd,MLPClassifier,64,32,30,0.0003,adam,0.3,True,0.0001,...,True,"[0, 1, 2, 3, 4]",1,0.613636,0.035722,0.597504,0.043479,0.614541,0.039402,f6309772107b448c87eba7f55ebf4f50
1,MLP_2_dropout_batchnorm,MLPClassifier,64,32,20,0.0010,adam,0.5,True,0.0000,...,False,"[0, 1, 2, 3, 4]",2,0.605313,0.016484,0.593966,0.020942,0.602573,0.016936,352c6e5f7a2f4a6fa5de06abc046724c
2,MLP_11_bigger_bn_dropout03,MLPClassifier,64,32,30,0.0003,adam,0.3,True,0.0001,...,True,"[0, 1, 2, 3, 4]",1,0.603943,0.033741,0.592233,0.034257,0.607282,0.036482,076d66bf64744962bd3de74324ad2722
3,MLP_9_dropout03_batchnorm_weightdecay,MLPClassifier,64,32,25,0.0010,adam,0.3,True,0.0010,...,True,"[0, 1, 2, 3, 4]",1,0.602486,0.030440,0.592171,0.032225,0.606360,0.030409,1a87dfa5a219407a9140d5b7c4d532d1
4,MLP_3_dropout_batchnorm_weightdecay,MLPClassifier,64,32,20,0.0010,adam,0.5,True,0.0001,...,False,"[0, 1, 2, 3, 4]",1,0.599699,0.018402,0.576510,0.031652,0.595174,0.024522,48b5b78fda804cba8dfa8c157d18652b
5,MLP_0_baseline,MLPClassifier,64,32,20,0.0010,adam,0.0,False,0.0000,...,False,"[0, 1, 2, 3, 4]",3,0.591366,0.022020,0.584379,0.024531,0.595886,0.026271,a4ad1833ab184b40abe1b66bff2e46ba
6,MLP_6_he_initialization,MLPClassifier,64,32,20,0.0010,adam,0.5,True,0.0001,...,False,"[0, 1, 2, 3, 4]",1,0.559246,0.022299,0.549667,0.024553,0.558648,0.023259,e6b9a6a1aeb347dcb182f52c0f70c7c1
7,MLP_4_data_augmentation,MLPClassifier,64,32,20,0.0010,adam,0.5,True,0.0001,...,False,"[0, 1, 2, 3, 4]",1,0.556469,0.015438,0.540410,0.022030,0.553902,0.018236,f18a6bbbbd2641eab166d5911c8c1355
8,MLP_5_xavier_initialization,MLPClassifier,64,32,20,0.0010,adam,0.5,True,0.0001,...,False,"[0, 1, 2, 3, 4]",3,0.548135,0.010302,0.533751,0.008584,0.550339,0.010668,912cda7dcace487089936161ec2b6d25
9,MLP_8_early_stopping,MLPClassifier,64,32,30,0.0010,adam,0.5,True,0.0001,...,True,"[0, 1, 2, 3, 4]",1,0.543871,0.026188,0.526526,0.035515,0.540734,0.028060,f1b6560f25454ca797f030c515be54eb


Comparación MLP guardada en: experiments\experiments_mlp.csv


In [6]:
# ============================================================
# 8. Mejor configuración
# ============================================================
#
# Esta configuración NO se evalúa todavía en test.
# El test fijo se reserva para el final del TP, después de comparar
# MLP y CNN.
# ============================================================

best_mlp = summary_df.iloc[0]

print("Mejor experimento MLP según val_accuracy_mean:")
display(best_mlp)

Mejor experimento MLP según val_accuracy_mean:


experiment                     MLP_10_lower_lr_dropout03_bn_wd
model                                            MLPClassifier
image_size                                                  64
batch_size                                                  32
epochs                                                      30
lr                                                      0.0003
optimizer                                                 adam
dropout                                                    0.3
batch_norm                                                True
weight_decay                                            0.0001
augmentation                                           minimal
initialization                                              he
hidden_dims                                         [512, 128]
early_stopping                                            True
folds_run                                      [0, 1, 2, 3, 4]
best_fold                                              